<a href="https://colab.research.google.com/github/ITATG/BBS_Project_work/blob/jad_jamous/BBS_project_01_database_checks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install pymssql

In [2]:
import time, pymssql
from google.colab import userdata

def connect(retries=10, wait=30):
    for i in range(retries):
        try:
            return pymssql.connect(
                server="crm-data.database.windows.net",
                user=userdata.get("AZURE_SQL_USER"),
                password=userdata.get("AZURE_SQL_PASSWORD"),
                database="rawdata-test",
                port=1433,
                login_timeout=60,
            )
        except pymssql.OperationalError as e:
            if "40613" in str(e) and i < retries - 1:
                print(f"DB waking up, retry {i+1}/{retries} in {wait}s…")
                time.sleep(wait)
            else:
                raise

conn = connect()

print("Connected successfully!")

Connected successfully!


In [3]:
import pandas as pd

query = """
SELECT TABLE_SCHEMA, TABLE_NAME
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_TYPE = 'BASE TABLE'
ORDER BY TABLE_SCHEMA, TABLE_NAME;
"""

with conn.cursor(as_dict=True) as cursor:
    cursor.execute(query)
    tables = pd.DataFrame(cursor.fetchall())

display(tables)

,TABLE_SCHEMA,TABLE_NAME
0,dbo,EQUIPMENT
1,dbo,PRODUCT
2,dbo,SCH
3,dbo,SCI
4,dbo,SKILLS


In [4]:
import pandas as pd

def q(sql):
    with conn.cursor(as_dict=True) as c:
        c.execute(sql)
        return pd.DataFrame(c.fetchall())

print("Connected as / to:")
display(q("SELECT SUSER_SNAME() AS login_name, USER_NAME() AS db_user, DB_NAME() AS db"))

print("My roles in this DB:")
display(q("""
SELECT r.name AS role
FROM sys.database_role_members m
JOIN sys.database_principals r ON r.principal_id = m.role_principal_id
WHERE m.member_principal_id = USER_ID()
"""))

print("Objects I can see (tables AND views):")
objs = q("SELECT TABLE_SCHEMA, TABLE_NAME, TABLE_TYPE FROM INFORMATION_SCHEMA.TABLES ORDER BY 1,2")
print(len(objs), "objects")
display(objs)

Connected as / to:


,login_name,db_user,db
0,bbs_PJ_team2,bbs_PJ_team2,rawdata-test


My roles in this DB:


,role
0,db_owner
1,db_datareader
2,db_datawriter


Objects I can see (tables AND views):
6 objects


,TABLE_SCHEMA,TABLE_NAME,TABLE_TYPE
0,dbo,EQUIPMENT,BASE TABLE
1,dbo,PRODUCT,BASE TABLE
2,dbo,SCH,BASE TABLE
3,dbo,SCI,BASE TABLE
4,dbo,SKILLS,BASE TABLE
5,sys,database_firewall_rules,VIEW


In [5]:
import pandas as pd

def q(sql):
    try:
        with conn.cursor(as_dict=True) as c:
            c.execute(sql)
            df = pd.DataFrame(c.fetchall())
            print(f"  -> {len(df)} rows")
            return df
    except Exception as e:
        print("  -> ERROR:", str(e)[:200])
        return pd.DataFrame()

print("1. Can I read data anywhere in this DB? (1 = yes, 0 = no)")
display(q("""
SELECT HAS_PERMS_BY_NAME(DB_NAME(), 'DATABASE', 'SELECT')          AS can_select_db,
       HAS_PERMS_BY_NAME(DB_NAME(), 'DATABASE', 'VIEW DEFINITION') AS can_view_definitions
"""))

print("2. Every permission I hold in this DB:")
display(q("SELECT permission_name, subentity_name FROM fn_my_permissions(NULL, 'DATABASE')"))

print("3. Schemas I can see (a team schema like 'team2' would show up here):")
display(q("SELECT name AS schema_name FROM sys.schemas ORDER BY name"))

print("4. Schema-level grants to me:")
display(q("""
SELECT s.name AS schema_name, p.permission_name, p.state_desc
FROM sys.database_permissions p
JOIN sys.schemas s ON p.class = 3 AND p.major_id = s.schema_id
WHERE p.grantee_principal_id = USER_ID()
"""))

print("5. Any user objects at all (tables, views, procedures):")
display(q("""
SELECT SCHEMA_NAME(schema_id) AS schema_name, name, type_desc
FROM sys.objects
WHERE is_ms_shipped = 0
ORDER BY 1, 2
"""))

1. Can I read data anywhere in this DB? (1 = yes, 0 = no)
  -> 1 rows


,can_select_db,can_view_definitions
0,1,1


2. Every permission I hold in this DB:
  -> 108 rows


,permission_name,subentity_name
0,CREATE TABLE,
1,CREATE VIEW,
2,CREATE PROCEDURE,
3,CREATE FUNCTION,
4,CREATE RULE,
...,...,...
103,VIEW DATABASE SECURITY AUDIT,
104,ALTER LEDGER CONFIGURATION,
105,ALTER ANY EXTERNAL MIRROR,
106,ALTER ANY INFORMATION PROTECTION,


3. Schemas I can see (a team schema like 'team2' would show up here):
  -> 13 rows


,schema_name
0,db_accessadmin
1,db_backupoperator
2,db_datareader
3,db_datawriter
4,db_ddladmin
5,db_denydatareader
6,db_denydatawriter
7,db_owner
8,db_securityadmin
9,dbo


4. Schema-level grants to me:
  -> 0 rows


""


5. Any user objects at all (tables, views, procedures):
  -> 10 rows


,schema_name,name,type_desc
0,dbo,EQUIPMENT,USER_TABLE
1,dbo,PK_EQUIPMENT,PRIMARY_KEY_CONSTRAINT
2,dbo,PK_PRODUCT,PRIMARY_KEY_CONSTRAINT
3,dbo,PK_SCH,PRIMARY_KEY_CONSTRAINT
4,dbo,PK_SCI,PRIMARY_KEY_CONSTRAINT
5,dbo,PK_SKILLS,PRIMARY_KEY_CONSTRAINT
6,dbo,PRODUCT,USER_TABLE
7,dbo,SCH,USER_TABLE
8,dbo,SCI,USER_TABLE
9,dbo,SKILLS,USER_TABLE


In [6]:
query = """
SELECT
    TABLE_SCHEMA,
    TABLE_NAME,
    COLUMN_NAME,
    ORDINAL_POSITION,
    DATA_TYPE,
    IS_NULLABLE,
    CHARACTER_MAXIMUM_LENGTH
FROM INFORMATION_SCHEMA.COLUMNS
ORDER BY
    TABLE_SCHEMA,
    TABLE_NAME,
    ORDINAL_POSITION;
"""

columns = pd.read_sql_query(query, conn)
display(columns)

/tmp/ipykernel_640/1577861596.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  columns = pd.read_sql_query(query, conn)


,TABLE_SCHEMA,TABLE_NAME,COLUMN_NAME,ORDINAL_POSITION,DATA_TYPE,IS_NULLABLE,CHARACTER_MAXIMUM_LENGTH
0,dbo,EQUIPMENT,EquipmentCode,1,varchar,NO,50.0
1,dbo,EQUIPMENT,ProductCode,2,varchar,YES,50.0
2,dbo,EQUIPMENT,ConstructionYear,3,int,YES,NaN
3,dbo,EQUIPMENT,InstallerCode,4,varchar,YES,20.0
4,dbo,EQUIPMENT,ManufacturerCode,5,varchar,YES,10.0
5,dbo,EQUIPMENT,PowerCode,6,varchar,YES,20.0
6,dbo,EQUIPMENT,FeukoLastYear,7,int,YES,NaN
7,dbo,EQUIPMENT,WarrantyEndDate,8,date,YES,NaN
8,dbo,EQUIPMENT,StartUpDate,9,date,YES,NaN
9,dbo,EQUIPMENT,WarrantyStartDate,10,date,YES,NaN


In [7]:
query = """
SELECT
    s.name AS schema_name,
    t.name AS table_name,
    c.name AS column_name,
    ic.key_ordinal
FROM sys.key_constraints kc
JOIN sys.tables t
    ON kc.parent_object_id = t.object_id
JOIN sys.schemas s
    ON t.schema_id = s.schema_id
JOIN sys.index_columns ic
    ON ic.object_id = t.object_id
   AND ic.index_id = kc.unique_index_id
JOIN sys.columns c
    ON c.object_id = t.object_id
   AND c.column_id = ic.column_id
WHERE kc.type = 'PK'
ORDER BY s.name, t.name, ic.key_ordinal;
"""

primary_keys = pd.read_sql_query(query, conn)
display(primary_keys)

/tmp/ipykernel_640/3270493203.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  primary_keys = pd.read_sql_query(query, conn)


,schema_name,table_name,column_name,key_ordinal
0,dbo,EQUIPMENT,EquipmentCode,1
1,dbo,PRODUCT,MaterialCode,1
2,dbo,SCH,ServiceConfirmationHeader,1
3,dbo,SCI,ServiceConfirmationHeader,1
4,dbo,SCI,ServiceConfirmationItem,2
5,dbo,SKILLS,ResourceID,1
6,dbo,SKILLS,SkillCode,2


In [8]:
query = """
SELECT
    fk.name AS foreign_key_name,
    SCHEMA_NAME(child.schema_id) AS child_schema,
    child.name AS child_table,
    child_col.name AS child_column,
    SCHEMA_NAME(parent.schema_id) AS parent_schema,
    parent.name AS parent_table,
    parent_col.name AS parent_column
FROM sys.foreign_key_columns fkc
JOIN sys.foreign_keys fk
    ON fk.object_id = fkc.constraint_object_id
JOIN sys.tables child
    ON child.object_id = fkc.parent_object_id
JOIN sys.columns child_col
    ON child_col.object_id = fkc.parent_object_id
   AND child_col.column_id = fkc.parent_column_id
JOIN sys.tables parent
    ON parent.object_id = fkc.referenced_object_id
JOIN sys.columns parent_col
    ON parent_col.object_id = fkc.referenced_object_id
   AND parent_col.column_id = fkc.referenced_column_id
ORDER BY child.name, fk.name;
"""

foreign_keys = pd.read_sql_query(query, conn)
display(foreign_keys)

/tmp/ipykernel_640/488142224.py:26: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  foreign_keys = pd.read_sql_query(query, conn)


,foreign_key_name,child_schema,child_table,child_column,parent_schema,parent_table,parent_column


In [9]:
counts = []

for row in tables.itertuples(index=False):
    schema = row.TABLE_SCHEMA.replace("]", "]]")
    table = row.TABLE_NAME.replace("]", "]]")
    full_name = f"[{schema}].[{table}]"

    query = f"SELECT COUNT_BIG(*) AS row_count FROM {full_name}"
    count = pd.read_sql_query(query, conn).iloc[0, 0]

    counts.append({
        "schema": row.TABLE_SCHEMA,
        "table": row.TABLE_NAME,
        "row_count": int(count)
    })

row_counts = pd.DataFrame(counts)
display(row_counts.sort_values("row_count", ascending=False))

/tmp/ipykernel_640/3305768172.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  count = pd.read_sql_query(query, conn).iloc[0, 0]


,schema,table,row_count
3,dbo,SCI,108240
2,dbo,SCH,19865
0,dbo,EQUIPMENT,15460
4,dbo,SKILLS,2784
1,dbo,PRODUCT,1519


In [10]:
for row in tables.itertuples(index=False):
    schema = row.TABLE_SCHEMA.replace("]", "]]")
    table = row.TABLE_NAME.replace("]", "]]")
    full_name = f"[{schema}].[{table}]"

    print(f"Table: {row.TABLE_SCHEMA}.{row.TABLE_NAME}")

    query = f"SELECT TOP (5) * FROM {full_name}"
    sample = pd.read_sql_query(query, conn)

    display(sample)

Table: dbo.EQUIPMENT


/tmp/ipykernel_640/2285304511.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  sample = pd.read_sql_query(query, conn)


,EquipmentCode,ProductCode,ConstructionYear,InstallerCode,ManufacturerCode,PowerCode,FeukoLastYear,WarrantyEndDate,StartUpDate,WarrantyStartDate
0,E00000024,7427200,2005,None,ELC,52.0,2025,2006-05-30,None,2005-09-30
1,E00000107,170298,2006,81099971.0,ELC,25.0,2024,2007-02-24,2006-02-24,2006-02-24
2,E00000458,0EAT19003,2004,81073324.0,ELC,60.0,0,2005-11-22,2004-11-22,2004-11-17
3,E00001046,170347,2006,81075143.0,ELC,50.0,0,2007-05-24,2006-05-24,2006-03-24
4,E00001047,170349,2006,81075143.0,ELC,50.0,0,2007-09-01,2006-05-02,2006-03-24


Table: dbo.PRODUCT


/tmp/ipykernel_640/2285304511.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  sample = pd.read_sql_query(query, conn)


,MaterialCode,EnergyTypeCode,ProductFamilyCode,ProductTypeCode,SkillProfileCode,WarengroupCode,MaterialDescription,PowerMaxKW,PowerMinKW,PowerMaxLT,RefrigerantFluid
0,0321234,OIL,90 10 508,BURNER,BU4,None,NC16 LX201 T1 CIPAG,160,0,0,0
1,0E1011243,OIL,90 10 526,BURNER,BU7,1435.0,BRUC. BICOM. ELCO EK 8.700 GL-EUF,700,0,0,0
2,0E1011721,OIL,90 10 526,BURNER,BU7,1432.0,BRUC. BICOM. ELCO EK6.30G,3100,0,0,0
3,0E11002264,CYL,10 20 770,WATER_HEAT,WH1,3152.0,ACCUMULATORE BASAM. 1000 ELE S.ISOL.,0,1000,0,0
4,0E1301848,OIL,90 10 522,BURNER,BU3,9900.0,VECTRON G 01.55,55,0,0,0


... (5 outputs left out)

,ServiceConfirmationHeader,ServiceConfirmationItem,ItemTypeCode,StatusCode,CreationTimestamp,StartWork,EndWork,ActualDuration
0,9004995356,205,ZSCR,E0006,2026-06-01 06:28:55,2026-06-01 07:48:00,2026-06-01 08:28:00,40
1,9004995356,206,ZTRH,E0006,2026-06-01 06:28:55,2026-06-01 07:23:00,2026-06-01 07:48:00,25
2,9004995356,207,ZUNP,E0006,2026-06-01 06:28:55,2026-06-01 00:00:00,1900-01-01 00:00:00,15
3,9004995356,208,ZIND,E0006,2026-06-01 06:28:55,2026-06-01 00:00:00,1900-01-01 00:00:00,40
4,9004995357,50,ZSCC,E0004,2026-06-01 06:42:00,2026-06-01 07:58:00,2026-06-01 08:40:00,42


Table: dbo.SKILLS


/tmp/ipykernel_640/2285304511.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  sample = pd.read_sql_query(query, conn)


,ResourceID,TeamCode,SkillCode
0,7002175,SECH.E.T03,BO1_Group
1,7002175,SECH.E.T03,BO2_Group
2,7002175,SECH.E.T03,BU1_Group
3,7002175,SECH.E.T03,BU2_Group
4,7002175,SECH.E.T03,BU3_Group


In [11]:
def q(sql):
    try:
        with conn.cursor(as_dict=True) as c:
            c.execute(sql)
            return pd.DataFrame(c.fetchall())
    except Exception as e:
        print("ERROR executing query:", e)
        return pd.DataFrame()

print("1. Date coverage (decides whether a 30-day label is possible)")
display(q("""
SELECT MIN(CreationTimestamp) first_visit, MAX(CreationTimestamp) last_visit,
       DATEDIFF(day, MIN(CreationTimestamp), MAX(CreationTimestamp)) days_covered,
       COUNT(*) visits FROM dbo.SCH"""))
display(q("""SELECT FORMAT(CreationTimestamp,'yyyy-MM') month, COUNT(*) visits
             FROM dbo.SCH GROUP BY FORMAT(CreationTimestamp,'yyyy-MM') ORDER BY 1"""))

print("2. Join coverage")
display(q("""
SELECT
 SUM(CASE WHEN s.EquipmentCode IS NULL THEN 1 ELSE 0 END) sch_null_equipment,
 SUM(CASE WHEN e.EquipmentCode IS NOT NULL THEN 1 ELSE 0 END) sch_matched_equipment,
 SUM(CASE WHEN p.MaterialCode IS NOT NULL THEN 1 ELSE 0 END) sch_matched_product,
 SUM(CASE WHEN x.h IS NOT NULL THEN 1 ELSE 0 END) sch_with_sci_items,
 COUNT(*) total_sch
FROM dbo.SCH s
LEFT JOIN dbo.EQUIPMENT e ON e.EquipmentCode = s.EquipmentCode
LEFT JOIN dbo.PRODUCT p   ON p.MaterialCode  = s.MaterialCode
LEFT JOIN (SELECT DISTINCT ServiceConfirmationHeader h FROM dbo.SCI) x
       ON x.h = s.ServiceConfirmationHeader"""))

print("3. Technician -> SKILLS match, raw vs leading zeros stripped")
display(q("""
WITH t AS (SELECT DISTINCT ServiceTechnicianCode c FROM dbo.SCH WHERE ServiceTechnicianCode IS NOT NULL AND ServiceTechnicianCode <> ''),
     r AS (SELECT DISTINCT ResourceID FROM dbo.SKILLS)
SELECT COUNT(*) technicians,
 SUM(CASE WHEN EXISTS(SELECT 1 FROM r WHERE r.ResourceID = t.c) THEN 1 ELSE 0 END) match_raw,
 SUM(CASE
     WHEN PATINDEX('%[^0]%', t.c) > 0 AND EXISTS(
         SELECT 1 FROM r
         WHERE r.ResourceID = SUBSTRING(t.c, PATINDEX('%[^0]%', t.c), LEN(t.c))
     ) THEN 1 ELSE 0
 END) match_no_leading_zeros
FROM t"""))

print("4. Code frequencies (for the codebook)")
display(q("SELECT HeaderStatusCode, COUNT(*) n FROM dbo.SCH GROUP BY HeaderStatusCode ORDER BY n DESC"))
display(q("SELECT ItemTypeCode, COUNT(*) n FROM dbo.SCI GROUP BY ItemTypeCode ORDER BY n DESC"))

print("5. Rough repeat signal: visits followed by another visit on same equipment within 30 days")
display(q("""
WITH v AS (
 SELECT EquipmentCode, CreationTimestamp t,
        LEAD(CreationTimestamp) OVER (PARTITION BY EquipmentCode ORDER BY CreationTimestamp) next_t
 FROM dbo.SCH WHERE EquipmentCode IS NOT NULL)
SELECT COUNT(*) visits,
 SUM(CASE WHEN DATEDIFF(day, t, next_t) <= 30 THEN 1 ELSE 0 END) next_visit_within_30d,
 SUM(CASE WHEN DATEDIFF(day, t, next_t) <= 7  THEN 1 ELSE 0 END) next_visit_within_7d
FROM v"""))

1. Date coverage (decides whether a 30-day label is possible)


,first_visit,last_visit,days_covered,visits
0,2026-06-01 06:26:19,2026-06-30 21:31:37,29,19865


,month,visits
0,2026-06,19865


2. Join coverage


,sch_null_equipment,sch_matched_equipment,sch_matched_product,sch_with_sci_items,total_sch
0,2015,17837,17838,19032,19865


3. Technician -> SKILLS match, raw vs leading zeros stripped
ERROR executing query: (130, b'Cannot perform an aggregate function on an expression containing an aggregate or a subquery.DB-Lib error message 20018, severity 15:\nGeneral SQL Server error: Check messages from the SQL Server\nDB-Lib error message 20018, severity 15:\nGeneral SQL Server error: Check messages from the SQL Server\n')


... (2 outputs left out)

,HeaderStatusCode,n
0,E0004,16950
1,E0006,2505
2,E0005,409
3,E0001,1


,ItemTypeCode,n
0,ZRVM,28892
1,ZUNP,18497
2,ZIND,18497
3,ZTRH,16414
4,ZSCR,16306
5,ZGRX,1111
6,ZSCS,1045
7,ZTRZ,1034
8,ZRVF,1006
9,ZFKO,881


5. Rough repeat signal: visits followed by another visit on same equipment within 30 days


,visits,next_visit_within_30d,next_visit_within_7d
0,17850,2586,2070


In [12]:
print("3. Technician -> SKILLS match (fixed)")
display(q("""
WITH t AS (
  SELECT DISTINCT ServiceTechnicianCode c,
         SUBSTRING(ServiceTechnicianCode, PATINDEX('%[^0]%', ServiceTechnicianCode + '.'), 50) c_stripped
  FROM dbo.SCH
  WHERE ServiceTechnicianCode IS NOT NULL AND ServiceTechnicianCode <> ''),
r AS (SELECT DISTINCT ResourceID FROM dbo.SKILLS)
SELECT COUNT(*) technicians,
       COUNT(r1.ResourceID) match_raw,
       COUNT(r2.ResourceID) match_no_leading_zeros
FROM t
LEFT JOIN r r1 ON r1.ResourceID = t.c
LEFT JOIN r r2 ON r2.ResourceID = t.c_stripped"""))

print("6. Gap to next visit on same equipment, in days (0 = same day)")
display(q("""
WITH v AS (
  SELECT CreationTimestamp t,
         LEAD(CreationTimestamp) OVER (PARTITION BY EquipmentCode ORDER BY CreationTimestamp) next_t
  FROM dbo.SCH WHERE EquipmentCode IS NOT NULL)
SELECT DATEDIFF(day, t, next_t) gap_days, COUNT(*) n
FROM v WHERE next_t IS NOT NULL
GROUP BY DATEDIFF(day, t, next_t) ORDER BY gap_days"""))

print("7. How many visits have a complete 7-day follow-up window")
display(q("""
SELECT COUNT(*) total_with_equipment,
       SUM(CASE WHEN CreationTimestamp <= DATEADD(day, -7, '2026-06-30 23:59:59') THEN 1 ELSE 0 END) eligible_7d
FROM dbo.SCH WHERE EquipmentCode IS NOT NULL"""))

3. Technician -> SKILLS match (fixed)


,technicians,match_raw,match_no_leading_zeros
0,397,0,355


6. Gap to next visit on same equipment, in days (0 = same day)


,gap_days,n
0,0,955
1,1,404
2,2,136
3,3,162
4,4,110
5,5,100
6,6,93
7,7,110
8,8,67
9,9,50


7. How many visits have a complete 7-day follow-up window


,total_with_equipment,eligible_7d
0,17850,13869


In [13]:
display(q("""
SELECT FORMAT(CreationTimestamp,'yyyy-MM') AS month,
       COUNT(*)                            AS visits,
       COUNT(DISTINCT EquipmentCode)       AS equipment
FROM dbo.SCH
GROUP BY FORMAT(CreationTimestamp,'yyyy-MM')
ORDER BY month"""))

display(q("""
SELECT (SELECT COUNT(*) FROM dbo.SCH) AS sch, (SELECT COUNT(*) FROM dbo.SCI) AS sci,
       (SELECT COUNT(*) FROM dbo.EQUIPMENT) AS equipment, (SELECT COUNT(*) FROM dbo.PRODUCT) AS product,
       (SELECT COUNT(*) FROM dbo.SKILLS) AS skills,
       (SELECT COUNT(*) FROM (SELECT ServiceConfirmationHeader FROM dbo.SCH
                              GROUP BY ServiceConfirmationHeader HAVING COUNT(*) > 1) d) AS duplicate_visits"""))

,month,visits,equipment
0,2026-06,19865,15264


,sch,sci,equipment,product,skills,duplicate_visits
0,19865,108240,15460,1519,2784,0


I installed the Python library that lets Colab connect to our Azure SQL database.

I connected to the database rawdata-test on crm-data.database.windows.net using our team login, bbs_PJ_team2. The username and password are stored in Colab secrets, not in the notebook code. If the database is asleep because it has been idle, the cell waits and retries automatically.

I listed the five tables: SCH, with one row per service visit; SCI, with the activity lines within each visit; EQUIPMENT; PRODUCT; and SKILLS, which shows which technician has which skill.

I checked who we’re logged in as and what the login can do. At first, it could only connect and couldn’t see any tables. After Lorenzo granted access, it received full owner rights, meaning it can read, change, and delete anything. That’s a risk to keep in mind: no one should modify or delete tables from the notebook.

I listed every column in every table, along with its type.

I found each table’s primary key — the column that uniquely identifies each row.

I looked for formal links between tables, called foreign keys, and found none. So we had to test the links ourselves.

I counted the rows: SCI has 108,240; SCH has 19,865; EQUIPMENT has 15,460; SKILLS has 2,784; and PRODUCT has 1,519.

I displayed the first 5 rows of each table to get a sense of what the data looks like.

I ran the main data checks:

Dates: All visits fall between 1 and 30 June 2026, so this data covers one month. Lorenzo has since received data covering more than one month, so we need to re-run these checks on it.

Links between tables: 99.9% of visits with an equipment code match the equipment table. About 10% of visits (2,015) have no equipment code, about 90% match a product, and 96% have activity lines.

Codes: I counted every visit status code and activity code to help build a codebook. Two activity codes, ZUNP and ZIND, appear exactly the same number of times; they’re probably added automatically to most visits, so they can’t tell us whether a visit was planned.

Repeat visits: Of the 17,850 visits with equipment, 2,586 (14.5%) were followed by another visit to the same equipment later in June, and 80% of those follow-up visits came within 7 days.
I fixed one check that failed and added two more:

Technicians: Once leading zeros are removed, technician codes in the visit table match the skills table for 355 of 397 technicians.

Time to the next visit, day by day: 955 repeat visits happen on the same day, which may mean one visit was recorded twice rather than that there was a real return visit.

Data available for a 7-day model: 13,869 visits have a full 7 days of data after them, so they can be used to train a model that predicts repeats within 7 days.

What this means: With only one month of data, we could only predict repeat visits within 7 days, not 30. We’re also still missing the order type — planned maintenance vs repair. Next, I’ll re-run these checks on Lorenzo’s longer dataset and create the shared database views: cleaned, correctly joined tables that we can all use to build the model.

In [14]:
# =====================================================================
# BBS predictive maintenance - analytics views
# Paste into ONE Colab cell, after the cell that creates `conn`.
#
# What it does
#   Creates a separate schema called `analytics` and four views in it.
#   Views are saved queries: they store no data and change nothing in
#   the raw tables (dbo.SCH, dbo.SCI, dbo.EQUIPMENT, dbo.PRODUCT,
#   dbo.SKILLS). Dropping a view never deletes data.
#
#   1. analytics.vw_product_clean    PRODUCT with the swapped power
#                                    columns fixed and 0 -> NULL
#   2. analytics.vw_equipment_clean  EQUIPMENT with ".0" codes and
#                                    0 placeholders cleaned
#   3. analytics.vw_intervention     ONE ROW PER VISIT: SCH + activity
#                                    items rolled up + technician +
#                                    equipment features
#   4. analytics.vw_repeat_7d        ONE ROW PER VISIT WITH EQUIPMENT:
#                                    next visit, gap, 7-day labels
#
# Safe to re-run: CREATE OR ALTER replaces the view definitions.
# =====================================================================

import pandas as pd

statements = [

# --- 0. schema -------------------------------------------------------
"""
IF SCHEMA_ID('analytics') IS NULL
    EXEC('CREATE SCHEMA analytics');
""",

# --- 1. product, with the load-script column swap undone --------------
# The load script put "Nr Power Max Lt" into PowerMinKW and
# "Nr Power Min Kw" into PowerMaxLT. This view swaps them back.
# 0 is treated as "not recorded".
"""
CREATE OR ALTER VIEW analytics.vw_product_clean AS
SELECT
    MaterialCode,
    EnergyTypeCode,
    ProductFamilyCode,
    ProductTypeCode,
    SkillProfileCode,
    CASE WHEN WarengroupCode LIKE '%.0'
         THEN LEFT(WarengroupCode, LEN(WarengroupCode) - 2)
         ELSE WarengroupCode END              AS WarengroupCode,
    MaterialDescription,
    NULLIF(PowerMaxKW, 0)                     AS PowerMaxKW,
    NULLIF(PowerMaxLT, 0)                     AS PowerMinKW,      -- column actually holds Min kW
    NULLIF(PowerMinKW, 0)                     AS CapacityMaxLitres, -- column actually holds Max litres
    NULLIF(RefrigerantFluid, '0')             AS RefrigerantFluid
FROM dbo.PRODUCT;
""",

# --- 2. equipment ------------------------------------------------------
"""
CREATE OR ALTER VIEW analytics.vw_equipment_clean AS
SELECT
    EquipmentCode,
    ProductCode,
    CASE WHEN ConstructionYear BETWEEN 1950 AND 2026
         THEN ConstructionYear END            AS ConstructionYear,
    CASE WHEN InstallerCode LIKE '%.0'
         THEN LEFT(InstallerCode, LEN(InstallerCode) - 2)
         ELSE InstallerCode END               AS InstallerCode,
    ManufacturerCode,
    CASE WHEN PowerCode LIKE '%.0'
         THEN LEFT(PowerCode, LEN(PowerCode) - 2)
         ELSE PowerCode END                   AS PowerCode,
    CASE WHEN FeukoLastYear BETWEEN 1990 AND 2026
         THEN FeukoLastYear END               AS CombustionCheckYear,  -- "Feuko" = Feuerungskontrolle (assumed)
    WarrantyStartDate,
    WarrantyEndDate,
    StartUpDate
FROM dbo.EQUIPMENT;
""",

# --- 3. one row per visit ----------------------------------------------
# Activity items are rolled up BEFORE joining, so visit counts are never
# inflated. EndWork = 1900-01-01 is treated as "no real time logged".
# Technician code: leading zeros removed, 'nan' (an artefact of the load
# script) treated as missing.
"""
CREATE OR ALTER VIEW analytics.vw_intervention AS
WITH items AS (
    SELECT
        ServiceConfirmationHeader,
        COUNT(*)                                                   AS n_items,
        COUNT(DISTINCT ItemTypeCode)                               AS n_item_types,
        SUM(ActualDuration)                                        AS total_minutes,
        SUM(CASE WHEN ItemTypeCode = 'ZTRH' THEN ActualDuration END) AS ztrh_minutes,
        SUM(CASE WHEN ItemTypeCode = 'ZSCR' THEN ActualDuration END) AS zscr_minutes,
        MIN(CASE WHEN EndWork > '1901-01-01' THEN StartWork END)   AS first_work_start,
        MAX(CASE WHEN EndWork > '1901-01-01' THEN EndWork END)     AS last_work_end
    FROM dbo.SCI
    GROUP BY ServiceConfirmationHeader
),
tech AS (
    SELECT ResourceID, MAX(TeamCode) AS TeamCode, COUNT(*) AS n_skills
    FROM dbo.SKILLS
    GROUP BY ResourceID
)
SELECT
    s.ServiceConfirmationHeader                    AS intervention_id,
    s.EquipmentCode,
    s.CreationTimestamp                            AS created_at,
    CAST(s.CreationTimestamp AS date)              AS created_date,
    s.HeaderStatusCode,
    s.MaterialCode,
    s.EnergyTypeCode,
    s.CategoryCode,
    s.ProductTypeCode                              AS ProductFamilyCode,  -- loaded from "Cd Product Family"
    s.ProductType1,
    s.ProductType2,
    s.ProductFamilyDescription,
    s.SkillProfileCode,

    -- technician
    x.tech_id                                      AS TechnicianID,
    t.TeamCode,
    t.n_skills                                     AS technician_n_skills,
    CASE
        WHEN t.ResourceID IS NULL OR s.SkillProfileCode IS NULL THEN NULL
        WHEN EXISTS (SELECT 1 FROM dbo.SKILLS k
                     WHERE k.ResourceID = x.tech_id
                       AND k.SkillCode  = s.SkillProfileCode + '_Group') THEN 1
        ELSE 0
    END                                            AS tech_has_required_skill,

    -- equipment, measured at the visit date
    e.ConstructionYear,
    YEAR(s.CreationTimestamp) - e.ConstructionYear AS equipment_age_years,
    e.CombustionCheckYear,
    YEAR(s.CreationTimestamp) - e.CombustionCheckYear AS years_since_combustion_check,
    CASE
        WHEN e.WarrantyEndDate IS NULL THEN NULL
        WHEN CAST(s.CreationTimestamp AS date)
             BETWEEN ISNULL(e.WarrantyStartDate, '1900-01-01') AND e.WarrantyEndDate THEN 1
        ELSE 0
    END                                            AS under_warranty,

    -- activity items (known only AFTER the visit: do not use for a
    -- before-dispatch model)
    i.n_items,
    i.n_item_types,
    i.total_minutes,
    i.ztrh_minutes,
    i.zscr_minutes,
    i.first_work_start,
    i.last_work_end
FROM dbo.SCH s
CROSS APPLY (
    SELECT CASE
             WHEN NULLIF(NULLIF(s.ServiceTechnicianCode, ''), 'nan') IS NULL THEN NULL
             ELSE SUBSTRING(s.ServiceTechnicianCode,
                            PATINDEX('%[^0]%', s.ServiceTechnicianCode + '.'), 50)
           END AS tech_id
) x
LEFT JOIN items i                     ON i.ServiceConfirmationHeader = s.ServiceConfirmationHeader
LEFT JOIN tech t                      ON t.ResourceID = x.tech_id
LEFT JOIN analytics.vw_equipment_clean e ON e.EquipmentCode = s.EquipmentCode;
""",

# --- 4. repeat-visit labels ----------------------------------------------
# For every visit that has equipment: when is the next visit on the same
# equipment? Two versions: any next visit, and the next visit at least
# one calendar day later (excludes same-day records).
# followup_complete_7d = 1 when 7 full days of data exist after the visit;
# labels are NULL otherwise (unknown, NOT "no repeat").
"""
CREATE OR ALTER VIEW analytics.vw_repeat_7d AS
WITH cutoff AS (
    SELECT DATEADD(day, -6, CAST(MAX(CreationTimestamp) AS date)) AS last_eligible_exclusive
    FROM dbo.SCH
)
SELECT
    s.ServiceConfirmationHeader                                   AS intervention_id,
    s.EquipmentCode,
    s.CreationTimestamp                                           AS created_at,
    n_any.next_at                                                 AS next_visit_at,
    DATEDIFF(day, s.CreationTimestamp, n_any.next_at)             AS gap_days,
    n_1d.next_at                                                  AS next_visit_after_same_day_at,
    DATEDIFF(day, s.CreationTimestamp, n_1d.next_at)              AS gap_days_excl_same_day,
    CASE WHEN s.CreationTimestamp < c.last_eligible_exclusive THEN 1 ELSE 0 END AS followup_complete_7d,
    CASE
        WHEN s.CreationTimestamp >= c.last_eligible_exclusive THEN NULL
        WHEN DATEDIFF(day, s.CreationTimestamp, n_1d.next_at) BETWEEN 1 AND 7 THEN 1
        ELSE 0
    END                                                           AS repeat_1_7d,   -- recommended label
    CASE
        WHEN s.CreationTimestamp >= c.last_eligible_exclusive THEN NULL
        WHEN DATEDIFF(day, s.CreationTimestamp, n_any.next_at) BETWEEN 0 AND 7 THEN 1
        ELSE 0
    END                                                           AS repeat_0_7d    -- sensitivity: includes same day
FROM dbo.SCH s
CROSS JOIN cutoff c
OUTER APPLY (
    SELECT MIN(w.CreationTimestamp) AS next_at
    FROM dbo.SCH w
    WHERE w.EquipmentCode = s.EquipmentCode
      AND w.CreationTimestamp > s.CreationTimestamp
) n_any
OUTER APPLY (
    SELECT MIN(w.CreationTimestamp) AS next_at
    FROM dbo.SCH w
    WHERE w.EquipmentCode = s.EquipmentCode
      AND DATEDIFF(day, s.CreationTimestamp, w.CreationTimestamp) >= 1
) n_1d
WHERE s.EquipmentCode IS NOT NULL;
""",
]

with conn.cursor() as c:
    for sql in statements:
        c.execute(sql)
conn.commit()
print("Views created in schema 'analytics'.")

# --- checks: run these and compare with the numbers in the brief ----------
def q(sql):
    with conn.cursor(as_dict=True) as c:
        c.execute(sql)
        return pd.DataFrame(c.fetchall())

print("Row counts (expect 19,865 / 17,850 / 1,519 / 15,460):")
display(q("""
SELECT (SELECT COUNT(*) FROM analytics.vw_intervention)    AS interventions,
       (SELECT COUNT(*) FROM analytics.vw_repeat_7d)       AS visits_with_equipment,
       (SELECT COUNT(*) FROM analytics.vw_product_clean)   AS products,
       (SELECT COUNT(*) FROM analytics.vw_equipment_clean) AS equipment
"""))

print("7-day labels (expect about 13,869 eligible visits):")
display(q("""
SELECT SUM(followup_complete_7d)          AS eligible_7d,
       SUM(repeat_1_7d)                   AS repeats_1_7d,
       AVG(CAST(repeat_1_7d AS float))    AS repeat_rate_1_7d,
       SUM(repeat_0_7d)                   AS repeats_0_7d,
       AVG(CAST(repeat_0_7d AS float))    AS repeat_rate_0_7d
FROM analytics.vw_repeat_7d
"""))

print("Feature coverage on visits with equipment:")
display(q("""
SELECT COUNT(*)                                   AS visits,
       COUNT(equipment_age_years)                 AS with_age,
       COUNT(years_since_combustion_check)        AS with_combustion_check,
       COUNT(under_warranty)                      AS with_warranty_info,
       COUNT(TeamCode)                            AS with_technician_team,
       SUM(tech_has_required_skill)               AS tech_has_required_skill
FROM analytics.vw_intervention
WHERE EquipmentCode IS NOT NULL
"""))

Views created in schema 'analytics'.
Row counts (expect 19,865 / 17,850 / 1,519 / 15,460):


,interventions,visits_with_equipment,products,equipment
0,19865,17850,1519,15460


7-day labels (expect about 13,869 eligible visits):


,eligible_7d,repeats_1_7d,repeat_rate_1_7d,repeats_0_7d,repeat_rate_0_7d
0,13869,1058,0.076285,1676,0.120845


Feature coverage on visits with equipment:


,visits,with_age,with_combustion_check,with_warranty_info,with_technician_team,tech_has_required_skill
0,17850,16976,5221,17287,17419,15480


In [17]:
display(q("""
SELECT AVG(CAST(repeat_0_7d AS float)) AS repeat_rate_7d
FROM analytics.vw_repeat_7d
WHERE followup_complete_7d = 1;
"""))

,repeat_rate_7d
0,0.120845


**repeat_rate_7d = ~12.08%**

Among all the service visits where the equipment was known and there was a complete 7-day follow-up window (meaning we have historical data for at least 7 days after the visit), approximately 12.08% of them had a repeat visit to the same piece of equipment within 7 days.

In [ ]:
DROP VIEW IF EXISTS analytics.vw_repeat_7d;
DROP VIEW IF EXISTS analytics.vw_intervention;
DROP VIEW IF EXISTS analytics.vw_equipment_clean;
DROP VIEW IF EXISTS analytics.vw_product_clean;
DROP SCHEMA IF EXISTS analytics;